# Small Sample Learning for Defect Classification

**Semiconductor Solutions Challenge 2026 — Problem A | Intel Corporation**

This notebook implements a **Prototypical Network** with a **ConvNeXt-Tiny** backbone for few-shot defect classification in semiconductor manufacturing images.

**Pipeline:**
1. Setup & Install dependencies
2. Upload/mount dataset
3. Configuration
4. Dataset loading with image caching
5. Model definition (PrototypicalNet + FinetuneClassifier + FocalLoss)
6. Callbacks (ModelCheckpoint, EarlyStopping, CSVLogger)
7. **Stage 1**: Episodic prototypical training
8. **Stage 2**: Fine-tuning with Focal Loss
9. Evaluation & plots
10. Single-image inference

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q torch torchvision timm scikit-learn matplotlib pillow numpy

In [ ]:
import os
import sys
import time
import random
import csv
import heapq
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, accuracy_score,
)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Upload / Mount Dataset

**Option A — Google Drive (recommended):**
Upload `Data/` folder to your Google Drive, then mount it.

**Option B — Direct upload:**
Upload and unzip `dataset.zip` directly into Colab.

In [ ]:
# ── Option A: Mount Google Drive ──
# Upload your Data/ folder to Google Drive first, then run this cell.
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/Data"  # adjust path to your Data folder

# ── Option B: Upload dataset.zip directly ──
# from google.colab import files
# uploaded = files.upload()  # upload dataset.zip
# !unzip -q dataset.zip -d Data/
# DATA_DIR = "/content/Data"

# ── Option C: Local / already available ──
# If running locally or data is already in place, set the path here:
DATA_DIR = "Data"  # Change this to your dataset path

# Verify dataset
for cls in sorted(os.listdir(DATA_DIR)):
    cls_path = os.path.join(DATA_DIR, cls)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif'))])
        print(f"  {cls}: {count} images")

## 3. Configuration

In [ ]:
# ===================== CONFIGURATION =====================
# Paths
OUTPUT_DIR = "outputs"
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# Class names (folder names in Data/)
CLASS_NAMES = [
    "good", "defect1", "defect2", "defect3", "defect4",
    "defect5", "defect8", "defect9", "defect10",
]
NUM_CLASSES = len(CLASS_NAMES)

# Image settings
IMAGE_SIZE = 224
BACKBONE = "convnext_tiny.fb_in22k"

# Prototypical network training
N_WAY = NUM_CLASSES
K_SHOT_SUPPORT = 5
Q_QUERY = 5
NUM_EPISODES_TRAIN = 100
NUM_EPISODES_VAL = 20
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# Fine-tuning (stage 2)
FINETUNE_EPOCHS = 20
FINETUNE_LR = 5e-5
FINETUNE_BATCH_SIZE = 32

# Data split
VAL_RATIO = 0.2
TEST_RATIO = 0.1
RANDOM_SEED = 42

# Callbacks
EARLY_STOPPING_PATIENCE = 7
MIN_DELTA = 1e-4
CHECKPOINT_SAVE_TOP_K = 3

# Device — auto-detect
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Device: {DEVICE}")
print(f"Classes: {NUM_CLASSES}")
print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")

## 4. Dataset Loading, Caching & Transforms

In [ ]:
# ===================== IMAGE CACHE =====================
_IMAGE_CACHE = {}

def _cache_image(path):
    if path not in _IMAGE_CACHE:
        _IMAGE_CACHE[path] = Image.open(path).convert("RGB")
    return _IMAGE_CACHE[path]

def preload_images(samples):
    total = len(samples)
    print(f"Pre-loading {total} images into RAM...", end=" ", flush=True)
    for i, (path, _) in enumerate(samples):
        _cache_image(path)
        if (i + 1) % 1000 == 0:
            print(f"{i+1}/{total}", end=" ", flush=True)
    print("Done.")

def get_cached_image(path):
    if path in _IMAGE_CACHE:
        return _IMAGE_CACHE[path].copy()
    return Image.open(path).convert("RGB")

# ===================== DATA LOADING =====================
def get_all_samples():
    samples = []
    class_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
    for class_name in CLASS_NAMES:
        class_dir = os.path.join(DATA_DIR, class_name)
        if not os.path.isdir(class_dir):
            continue
        idx = class_to_idx[class_name]
        for fname in os.listdir(class_dir):
            if fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif")):
                samples.append((os.path.join(class_dir, fname), idx))
    return samples, class_to_idx

def split_dataset(samples, val_ratio=VAL_RATIO, test_ratio=TEST_RATIO, seed=RANDOM_SEED):
    paths, labels = zip(*samples)
    paths, labels = list(paths), list(labels)
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        paths, labels, test_size=val_ratio + test_ratio, stratify=labels, random_state=seed)
    relative_test = test_ratio / (val_ratio + test_ratio)
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=relative_test, stratify=temp_labels, random_state=seed)
    return list(zip(train_paths, train_labels)), list(zip(val_paths, val_labels)), list(zip(test_paths, test_labels))

# ===================== TRANSFORMS =====================
def get_train_transform():
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.2),
    ])

def get_val_transform():
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

# ===================== DATASET CLASS =====================
class DefectDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = get_cached_image(path)
        if self.transform:
            img = self.transform(img)
        return img, label

# ===================== EPISODIC SAMPLER =====================
class EpisodicSampler:
    def __init__(self, samples, n_way, k_shot, q_query, num_episodes):
        self.n_way, self.k_shot, self.q_query = n_way, k_shot, q_query
        self.num_episodes = num_episodes
        self.class_samples = defaultdict(list)
        for path, label in samples:
            self.class_samples[label].append(path)
        self.available_classes = [c for c, p in self.class_samples.items() if len(p) >= 2]

    def __len__(self):
        return self.num_episodes

    def __iter__(self):
        for _ in range(self.num_episodes):
            episode_classes = (self.available_classes[:] if len(self.available_classes) <= self.n_way
                               else random.sample(self.available_classes, self.n_way))
            support_paths, support_labels, query_paths, query_labels = [], [], [], []
            for i, cls in enumerate(episode_classes):
                cls_paths = self.class_samples[cls]
                n = len(cls_paths)
                k = min(self.k_shot, max(1, n // 2))
                q = min(self.q_query, n - k)
                if q < 1:
                    k = max(1, n - 1); q = n - k
                selected = random.sample(cls_paths, k + q)
                support_paths.extend(selected[:k]); support_labels.extend([i] * k)
                query_paths.extend(selected[k:k+q]); query_labels.extend([i] * q)
            yield support_paths, support_labels, query_paths, query_labels, episode_classes

def load_episode_images(paths, transform):
    images = [transform(get_cached_image(p)) if transform else get_cached_image(p) for p in paths]
    return torch.stack(images)

# ===================== DATA LOADERS =====================
def get_weighted_sampler(samples):
    labels = [s[1] for s in samples]
    cc = np.bincount(labels, minlength=NUM_CLASSES)
    cw = 1.0 / (cc + 1e-6)
    sw = [cw[l] for l in labels]
    return WeightedRandomSampler(sw, len(sw), replacement=True)

def get_finetune_loaders(train_samples, val_samples):
    train_ds = DefectDataset(train_samples, get_train_transform())
    val_ds = DefectDataset(val_samples, get_val_transform())
    sampler = get_weighted_sampler(train_samples)
    train_loader = DataLoader(train_ds, batch_size=FINETUNE_BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=FINETUNE_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader

print("Dataset utilities ready.")

## 5. Model Definitions

In [ ]:
class PrototypicalNet(nn.Module):
    """Prototypical Network: ConvNeXt-Tiny -> projection -> L2-normalized embedding."""

    def __init__(self, backbone_name=BACKBONE, embedding_dim=256, freeze_backbone=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        backbone_dim = self.backbone.num_features
        self.projector = nn.Sequential(
            nn.Linear(backbone_dim, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.2), nn.Linear(512, embedding_dim),
        )
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

    def forward(self, x):
        features = self.backbone(x)
        embeddings = self.projector(features)
        return F.normalize(embeddings, p=2, dim=-1)

    def compute_prototypes(self, support_embeddings, support_labels):
        n_way = support_labels.max().item() + 1
        prototypes = torch.zeros(n_way, support_embeddings.size(1), device=support_embeddings.device)
        for c in range(n_way):
            mask = support_labels == c
            prototypes[c] = support_embeddings[mask].mean(dim=0)
        return F.normalize(prototypes, p=2, dim=-1)

    def classify(self, query_embeddings, prototypes, temperature=10.0):
        return temperature * torch.mm(query_embeddings, prototypes.t())


class FinetuneClassifier(nn.Module):
    """Wraps prototypical backbone with a linear classification head."""

    def __init__(self, proto_net, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = proto_net.backbone
        self.projector = proto_net.projector
        embed_dim = proto_net.projector[-1].out_features
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        embeddings = F.normalize(self.projector(features), p=2, dim=-1)
        return self.classifier(embeddings)

    def get_embeddings(self, x):
        features = self.backbone(x)
        return F.normalize(self.projector(features), p=2, dim=-1)


class FocalLoss(nn.Module):
    """Focal loss for handling class imbalance."""

    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce_loss)
        return (((1 - pt) ** self.gamma) * ce_loss).mean()

print("Models defined: PrototypicalNet, FinetuneClassifier, FocalLoss")

## 6. Callbacks (Checkpoint, Early Stopping, Logger)

In [ ]:
class ModelCheckpoint:
    """Save top-K models by monitored metric."""

    def __init__(self, dirpath=MODEL_DIR, filename_prefix="checkpoint",
                 monitor="val_acc", mode="max", save_top_k=CHECKPOINT_SAVE_TOP_K):
        self.dirpath, self.filename_prefix = dirpath, filename_prefix
        self.monitor, self.mode, self.save_top_k = monitor, mode, save_top_k
        os.makedirs(dirpath, exist_ok=True)
        self._heap, self.best_score, self.best_path = [], None, None

    def _is_better(self, current, best):
        return current > best if self.mode == "max" else current < best

    def _score_key(self, score):
        return score if self.mode == "max" else -score

    def __call__(self, epoch, model, metrics):
        score = metrics[self.monitor]
        path = os.path.join(self.dirpath, f"{self.filename_prefix}_epoch{epoch:03d}_{self.monitor}={score:.4f}.pth")
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "metrics": metrics}, path)
        if self.best_score is None or self._is_better(score, self.best_score):
            self.best_score, self.best_path = score, path
            print(f"  [Checkpoint] New best {self.monitor}={score:.4f} -> {os.path.basename(path)}")
        hs = self._score_key(score)
        if len(self._heap) < self.save_top_k:
            heapq.heappush(self._heap, (hs, path))
        else:
            ws, wp = self._heap[0]
            if hs > ws:
                heapq.heapreplace(self._heap, (hs, path))
                if os.path.exists(wp): os.remove(wp)
            elif os.path.exists(path) and path != self.best_path:
                os.remove(path)

    def load_best(self, model, device=None):
        if self.best_path and os.path.exists(self.best_path):
            ckpt = torch.load(self.best_path, map_location=device, weights_only=True)
            model.load_state_dict(ckpt["model_state_dict"])
            print(f"  [Checkpoint] Restored best model ({self.monitor}={self.best_score:.4f})")
            return ckpt["metrics"]
        return None


class EarlyStopping:
    """Stop training when metric stops improving."""

    def __init__(self, patience=EARLY_STOPPING_PATIENCE, min_delta=MIN_DELTA, monitor="val_acc", mode="max"):
        self.patience, self.min_delta, self.monitor, self.mode = patience, min_delta, monitor, mode
        self.best_score, self.counter, self.should_stop, self.best_epoch = None, 0, False, 0

    def _is_improvement(self, current, best):
        return current > best + self.min_delta if self.mode == "max" else current < best - self.min_delta

    def __call__(self, epoch, metrics):
        score = metrics[self.monitor]
        if self.best_score is None or self._is_improvement(score, self.best_score):
            self.best_score, self.counter, self.best_epoch = score, 0, epoch
        else:
            self.counter += 1
            print(f"  [EarlyStopping] No improvement {self.counter}/{self.patience} "
                  f"(best {self.monitor}={self.best_score:.4f} @ epoch {self.best_epoch})")
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  [EarlyStopping] Stopping training.")


class CSVLogger:
    """Log metrics to CSV."""

    def __init__(self, log_dir=LOG_DIR, filename="training_log.csv"):
        os.makedirs(log_dir, exist_ok=True)
        self.filepath = os.path.join(log_dir, filename)

    def __call__(self, epoch, metrics):
        row = {"epoch": epoch, **metrics}
        write_header = not os.path.exists(self.filepath)
        with open(self.filepath, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=row.keys())
            if write_header: writer.writeheader()
            writer.writerow(row)

    def get_history(self):
        if not os.path.exists(self.filepath): return []
        with open(self.filepath, "r") as f:
            return [{k: float(v) if k != "epoch" else int(float(v)) for k, v in row.items()} for row in csv.DictReader(f)]


class CallbackRunner:
    """Orchestrates all callbacks."""

    def __init__(self, stage="proto"):
        self.checkpoint = ModelCheckpoint(filename_prefix=stage, monitor="val_acc", mode="max")
        self.early_stopping = EarlyStopping(monitor="val_acc", mode="max")
        self.csv_logger = CSVLogger(filename=f"{stage}_log.csv")

    @property
    def should_stop(self):
        return self.early_stopping.should_stop

    def on_epoch_end(self, epoch, model, metrics, optimizer=None):
        if optimizer:
            for i, g in enumerate(optimizer.param_groups):
                metrics[f"lr_{g.get('name', f'group_{i}')}"] = g["lr"]
        self.csv_logger(epoch, metrics)
        self.checkpoint(epoch, model, metrics)
        self.early_stopping(epoch, metrics)

    def load_best_model(self, model, device=None):
        return self.checkpoint.load_best(model, device)

print("Callbacks ready.")

## 7. Load Data & Pre-cache Images

In [ ]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device(DEVICE)
print(f"Using device: {device}")

# Load and split
samples, class_to_idx = get_all_samples()
print(f"\nTotal samples: {len(samples)}")
for name, idx in class_to_idx.items():
    count = sum(1 for _, l in samples if l == idx)
    print(f"  {name}: {count}")

train_samples, val_samples, test_samples = split_dataset(samples)
print(f"\nSplit: train={len(train_samples)}, val={len(val_samples)}, test={len(test_samples)}")

# Pre-load all images into RAM
all_samples = train_samples + val_samples + test_samples
preload_images(all_samples)

## 8. Visualize Sample Images

In [ ]:
# Show one sample from each class
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(20, 3))
class_samples_dict = defaultdict(list)
for path, label in samples:
    class_samples_dict[label].append(path)

for idx, name in enumerate(CLASS_NAMES):
    ax = axes[idx]
    paths = class_samples_dict[idx]
    if paths:
        img = Image.open(paths[0]).convert("L")  # show as grayscale
        ax.imshow(img, cmap="gray")
    ax.set_title(f"{name}\n({len(paths)})", fontsize=9)
    ax.axis("off")
plt.suptitle("One Sample Per Class", fontsize=14)
plt.tight_layout()
plt.show()

# Class distribution bar chart
counts = [len(class_samples_dict[i]) for i in range(NUM_CLASSES)]
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(CLASS_NAMES, counts, color=["#4CAF50"] + ["#F44336"] * 8)
for bar, c in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, str(c), ha="center", fontsize=9)
ax.set_ylabel("Number of Images")
ax.set_title("Class Distribution (Highly Imbalanced)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 9. Stage 1 — Prototypical (Episodic) Training

In [ ]:
def evaluate_prototypical(model, val_samples, transform, device):
    model.eval()
    sampler = EpisodicSampler(val_samples, N_WAY, K_SHOT_SUPPORT, Q_QUERY, NUM_EPISODES_VAL)
    total_acc, n = 0.0, 0
    with torch.no_grad():
        for support_paths, support_labels, query_paths, query_labels, _ in sampler:
            s_imgs = load_episode_images(support_paths, transform).to(device)
            q_imgs = load_episode_images(query_paths, transform).to(device)
            s_labels = torch.tensor(support_labels, device=device)
            q_labels = torch.tensor(query_labels, device=device)
            prototypes = model.compute_prototypes(model(s_imgs), s_labels)
            logits = model.classify(model(q_imgs), prototypes)
            total_acc += (logits.argmax(1) == q_labels).float().mean().item()
            n += 1
    return total_acc / max(n, 1)

def train_prototypical(model, train_samples, val_samples, device):
    print("\n" + "=" * 60)
    print("STAGE 1: Prototypical (Episodic) Training")
    print("=" * 60)

    optimizer = AdamW([
        {"params": model.backbone.parameters(), "lr": LEARNING_RATE * 0.1, "name": "backbone"},
        {"params": model.projector.parameters(), "lr": LEARNING_RATE, "name": "projector"},
    ], weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
    train_tf, val_tf = get_train_transform(), get_val_transform()
    callbacks = CallbackRunner(stage="proto")

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        model.train()
        ep_loss, ep_acc, n_ep = 0.0, 0.0, 0
        sampler = EpisodicSampler(train_samples, N_WAY, K_SHOT_SUPPORT, Q_QUERY, NUM_EPISODES_TRAIN)

        for s_paths, s_labels, q_paths, q_labels, _ in sampler:
            s_imgs = load_episode_images(s_paths, train_tf).to(device)
            q_imgs = load_episode_images(q_paths, train_tf).to(device)
            s_labels_t = torch.tensor(s_labels, device=device)
            q_labels_t = torch.tensor(q_labels, device=device)
            s_emb, q_emb = model(s_imgs), model(q_imgs)
            prototypes = model.compute_prototypes(s_emb, s_labels_t)
            logits = model.classify(q_emb, prototypes)
            loss = F.cross_entropy(logits, q_labels_t)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item(); ep_acc += (logits.argmax(1) == q_labels_t).float().mean().item(); n_ep += 1

        scheduler.step()
        avg_loss, avg_acc = ep_loss / n_ep, ep_acc / n_ep
        val_acc = evaluate_prototypical(model, val_samples, val_tf, device)
        et = time.time() - t0

        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | "
              f"Train Acc: {avg_acc:.4f} | Val Acc: {val_acc:.4f} | Time: {et:.1f}s")

        metrics = {"train_loss": round(avg_loss, 6), "train_acc": round(avg_acc, 6),
                   "val_acc": round(val_acc, 6), "epoch_time": round(et, 2)}
        callbacks.on_epoch_end(epoch, model, metrics, optimizer)
        if callbacks.should_stop:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

    callbacks.load_best_model(model, device)
    return model

# Run Stage 1
proto_model = PrototypicalNet(embedding_dim=256, freeze_backbone=False).to(device)
proto_model = train_prototypical(proto_model, train_samples, val_samples, device)

# Save
proto_path = os.path.join(MODEL_DIR, "proto_model.pth")
torch.save(proto_model.state_dict(), proto_path)
print(f"\nSaved prototypical model to {proto_path}")

## 10. Stage 2 — Fine-tuning with Focal Loss

In [ ]:
def train_finetune(proto_model, train_samples, val_samples, device):
    print("\n" + "=" * 60)
    print("STAGE 2: Fine-tuning with Focal Loss")
    print("=" * 60)

    classifier = FinetuneClassifier(proto_model).to(device)
    train_loader, val_loader = get_finetune_loaders(train_samples, val_samples)

    # Class weights for focal loss
    labels = [s[1] for s in train_samples]
    cc = np.bincount(labels, minlength=NUM_CLASSES).astype(np.float32)
    cw = 1.0 / (cc + 1e-6); cw = cw / cw.sum() * NUM_CLASSES
    alpha = torch.tensor(cw, device=device)
    criterion = FocalLoss(alpha=alpha, gamma=2.0)

    optimizer = AdamW([
        {"params": classifier.backbone.parameters(), "lr": FINETUNE_LR * 0.1, "name": "backbone"},
        {"params": classifier.projector.parameters(), "lr": FINETUNE_LR, "name": "projector"},
        {"params": classifier.classifier.parameters(), "lr": FINETUNE_LR, "name": "head"},
    ], weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=FINETUNE_EPOCHS, eta_min=1e-6)
    callbacks = CallbackRunner(stage="finetune")

    for epoch in range(1, FINETUNE_EPOCHS + 1):
        t0 = time.time()
        classifier.train()
        total_loss, correct, total = 0.0, 0, 0
        for images, labels_b in train_loader:
            images, labels_b = images.to(device), labels_b.to(device)
            logits = classifier(images)
            loss = criterion(logits, labels_b)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(classifier.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(1) == labels_b).sum().item()
            total += images.size(0)

        scheduler.step()
        classifier.eval()
        vc, vt, vl = 0, 0, 0.0
        with torch.no_grad():
            for images, labels_b in val_loader:
                images, labels_b = images.to(device), labels_b.to(device)
                logits = classifier(images)
                vc += (logits.argmax(1) == labels_b).sum().item()
                vt += images.size(0)
                vl += criterion(logits, labels_b).item() * images.size(0)

        tr_acc, val_acc = correct / total, vc / vt
        tr_loss, val_loss = total_loss / total, vl / vt
        et = time.time() - t0

        print(f"Epoch {epoch:3d}/{FINETUNE_EPOCHS} | Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Time: {et:.1f}s")

        metrics = {"train_loss": round(tr_loss, 6), "train_acc": round(tr_acc, 6),
                   "val_loss": round(val_loss, 6), "val_acc": round(val_acc, 6), "epoch_time": round(et, 2)}
        callbacks.on_epoch_end(epoch, classifier, metrics, optimizer)
        if callbacks.should_stop:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

    callbacks.load_best_model(classifier, device)
    return classifier

# Run Stage 2
classifier = train_finetune(proto_model, train_samples, val_samples, device)

# Save
ft_path = os.path.join(MODEL_DIR, "finetune_model.pth")
torch.save(classifier.state_dict(), ft_path)
print(f"\nSaved fine-tuned model to {ft_path}")

# Save data splits
torch.save({"test": test_samples, "train": train_samples, "val": val_samples},
           os.path.join(OUTPUT_DIR, "test_samples.pth"))
print("Saved data splits.")

## 11. Evaluation — Metrics & Plots

In [ ]:
# ===================== EVALUATE ON TEST SET =====================
classifier.eval()
transform = get_val_transform()
test_ds = DefectDataset(test_samples, transform)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = classifier(images)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

preds = np.array(all_preds)
labels = np.array(all_labels)

overall_acc = accuracy_score(labels, preds)
print(f"\nOverall Test Accuracy: {overall_acc:.4f} ({overall_acc * 100:.1f}%)")
print(f"\nClassification Report:")
present = sorted(set(labels))
print(classification_report(labels, preds, target_names=[CLASS_NAMES[i] for i in present], digits=4))

### 11a. Accuracy vs Class Occurrence

In [ ]:
# Accuracy vs Occurrence
classes = sorted(set(labels))
c_acc = [(preds[labels == c] == c).mean() * 100 for c in classes]
c_cnt = [(labels == c).sum() for c in classes]
c_names = [CLASS_NAMES[c] for c in classes]

fig, ax1 = plt.subplots(figsize=(12, 6))
x = np.arange(len(classes)); w = 0.35
b1 = ax1.bar(x - w/2, c_acc, w, label="Accuracy (%)", color="#2196F3", alpha=0.8)
ax1.set_ylabel("Accuracy (%)"); ax1.set_ylim(0, 105)
ax2 = ax1.twinx()
b2 = ax2.bar(x + w/2, c_cnt, w, label="Sample Count", color="#FF9800", alpha=0.8)
ax2.set_ylabel("Test Samples")
ax1.set_xticks(x); ax1.set_xticklabels(c_names, rotation=45, ha="right")
ax1.set_title("Classification Accuracy vs Defect Class Occurrence", fontsize=14)
for bar, v in zip(b1, c_acc):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f"{v:.1f}%", ha="center", fontsize=9)
for bar, v in zip(b2, c_cnt):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, str(v), ha="center", fontsize=9)
l1, la1 = ax1.get_legend_handles_labels(); l2, la2 = ax2.get_legend_handles_labels()
ax1.legend(l1+l2, la1+la2, loc="lower right")
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "accuracy_vs_occurrence.png"), dpi=150); plt.show()

### 11b. Confusion Matrix

In [ ]:
# Confusion Matrix
present = sorted(set(labels) | set(preds))
disp_labels = [CLASS_NAMES[c] for c in present]
cm = confusion_matrix(labels, preds, labels=present)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm, display_labels=disp_labels).plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "confusion_matrix.png"), dpi=150); plt.show()

### 11c. Per-Class F1 Score

In [ ]:
# Per-Class F1
present = sorted(set(labels))
f1_vals = f1_score(labels, preds, labels=present, average=None)
disp_labels = [CLASS_NAMES[c] for c in present]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(disp_labels, f1_vals * 100, color="#4CAF50", alpha=0.8)
for bar, v in zip(bars, f1_vals * 100):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f"{v:.1f}%", ha="center", fontsize=10)
ax.set_ylabel("F1 Score (%)"); ax.set_xlabel("Defect Class")
ax.set_title("Per-Class F1 Score", fontsize=14); ax.set_ylim(0, 105)
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "per_class_f1.png"), dpi=150); plt.show()

### 11d. Learning Curve — How Quickly the Model Learns

In [ ]:
# Learning curve: accuracy vs K-shot
transform = get_val_transform()
class_train = defaultdict(list)
for path, label in train_samples:
    class_train[label].append(path)
class_test = defaultdict(list)
for path, label in test_samples:
    class_test[label].append(path)

k_values = [1, 2, 3, 5, 10, 15, 20]
accuracies = []
n_trials = 5

proto_model.eval()
with torch.no_grad():
    for k in k_values:
        trial_accs = []
        for _ in range(n_trials):
            all_p, all_l = [], []
            s_paths, s_labels, avail = [], [], []
            for c in sorted(class_train.keys()):
                paths = class_train[c]
                ak = min(k, len(paths))
                sel = random.sample(paths, ak)
                s_paths.extend(sel); s_labels.extend([c] * ak); avail.append(c)

            s_imgs = load_episode_images(s_paths, transform).to(device)
            s_emb = proto_model(s_imgs)
            lmap = {c: i for i, c in enumerate(avail)}
            mapped = torch.tensor([lmap[l] for l in s_labels], device=device)
            prototypes = proto_model.compute_prototypes(s_emb, mapped)

            for c in avail:
                if not class_test.get(c): continue
                t_imgs = load_episode_images(class_test[c], transform).to(device)
                logits = proto_model.classify(proto_model(t_imgs), prototypes)
                ps = logits.argmax(1).cpu().numpy()
                rmap = {i: c for c, i in lmap.items()}
                all_p.extend([rmap[p] for p in ps])
                all_l.extend([c] * len(class_test[c]))

            trial_accs.append(accuracy_score(all_l, all_p))
        m, s = np.mean(trial_accs), np.std(trial_accs)
        accuracies.append((m, s))
        print(f"  K={k:3d} shots: {m:.4f} +/- {s:.4f}")

means = [a[0]*100 for a in accuracies]
stds = [a[1]*100 for a in accuracies]

fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(k_values, means, yerr=stds, marker="o", capsize=5, linewidth=2, markersize=8, color="#2196F3")
ax.fill_between(k_values, [m-s for m,s in zip(means,stds)], [m+s for m,s in zip(means,stds)], alpha=0.2, color="#2196F3")
ax.axhline(y=85, color="r", linestyle="--", alpha=0.7, label="85% target")
ax.set_xlabel("Number of Support Examples per Class (K-shot)", fontsize=12)
ax.set_ylabel("Test Accuracy (%)", fontsize=12)
ax.set_title("Learning Speed: Accuracy vs Number of Examples", fontsize=14)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3); ax.set_ylim(0, 100)
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "learning_curve.png"), dpi=150); plt.show()

### 11e. Inference Time Measurement

In [ ]:
# Measure inference time
dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
classifier.eval()

# Warm up
with torch.no_grad():
    for _ in range(10):
        classifier(dummy)

times = []
with torch.no_grad():
    for _ in range(50):
        t0 = time.time()
        classifier(dummy)
        times.append(time.time() - t0)

avg_ms = np.mean(times) * 1000
print(f"Average inference time: {avg_ms:.1f} ms per image")
print(f"Well within the ~1 second target ({avg_ms/1000:.3f}s).")

## 12. Single-Image Inference

In [ ]:
def predict_image(image_path, model=classifier, device=device):
    """Classify a single image and display results."""
    transform = get_val_transform()
    img = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

    pred_idx = probs.argmax()
    pred_class = CLASS_NAMES[pred_idx]
    confidence = probs[pred_idx]

    # Display
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.imshow(Image.open(image_path).convert("L"), cmap="gray")
    ax1.set_title(f"Prediction: {pred_class} ({confidence:.2%})", fontsize=13)
    ax1.axis("off")

    colors = ["#4CAF50" if i == pred_idx else "#90CAF9" for i in range(NUM_CLASSES)]
    ax2.barh(CLASS_NAMES, probs * 100, color=colors)
    ax2.set_xlabel("Confidence (%)")
    ax2.set_title("Class Probabilities")
    ax2.set_xlim(0, 105)
    plt.tight_layout()
    plt.show()

    return pred_class, confidence

# ── Test on a sample image ──
# Change this path to any image you want to classify
sample_path = test_samples[0][0]
true_label = CLASS_NAMES[test_samples[0][1]]
pred, conf = predict_image(sample_path)
print(f"True: {true_label} | Predicted: {pred} ({conf:.2%})")

## 13. Download Models (Colab)

Run this cell to download trained models from Colab to your local machine.

In [ ]:
# Download trained models and plots (only works in Colab)
try:
    from google.colab import files
    # Zip outputs
    import shutil
    shutil.make_archive("outputs", "zip", ".", "outputs")
    files.download("outputs.zip")
    print("Download started!")
except ImportError:
    print("Not running in Colab. Models saved in outputs/ directory.")
    print(f"  Models: {MODEL_DIR}/")
    print(f"  Plots:  {PLOT_DIR}/")
    print(f"  Logs:   {LOG_DIR}/")